# ⚾ NRFI/YRFI Model v3
### 3-Year Dataset · Full-Game Rolling Stats · xFIP/SIERA · Calibrated Probabilities

---
## Why v2 failed + what v3 fixes

| Problem in v2 | Root cause | v3 fix |
|---|---|---|
| p_bot had zero signal (corr=0.001) | 1st-inning only = ~30 PA per rolling window (pure noise) | Full-game rolling stats = ~250 PA per window |
| Both models output ~0.5 | Features too weak to separate games | xFIP + SIERA + K-BB% (luck-independent quality) |
| Only 530 test games | Single season + rolling filter | 3 seasons (2022-2024) = ~2,900 test games |
| Independence formula blew up | Combining two noisy signals amplified noise | Single unified model with explicit home/away features |
| Probabilities not calibrated | Raw logistic regression output | CalibratedClassifierCV (isotonic) |

## Feature set
```
HOME PITCHER:  roll_xfip, roll_kbb_pct, roll_k_pct, roll_bb_pct, prior_yr_xfip
AWAY PITCHER:  roll_xfip, roll_kbb_pct, roll_k_pct, roll_bb_pct, prior_yr_xfip
HOME BATTING:  leadoff_obp, team_k_pct, team_woba
AWAY BATTING:  leadoff_obp, team_k_pct, team_woba
CONTEXT:       park_factor, weather_offense_factor
```
**Runtime:** First run ~45-60 min (downloads 3 seasons ~2.4GB). Cached after that.


## 1 · Install & Setup

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score,
                              brier_score_loss)

pb.cache.enable()
warnings.filterwarnings('ignore')
os.makedirs('cache', exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
SEASONS      = [2022, 2023, 2024]
ROLLING_N    = 10    # starts for rolling pitcher stats
ROLLING_BATT = 50    # PA window for leadoff OBP

PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'ARI':99,'MIA':98,'NYM':98,
    'OAK':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}
WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}

# v3 unified feature set
FEATURES = [
    # Home pitcher (faces away batters)
    'home_roll_k_pct',
    'home_roll_bb_pct',
    'home_roll_kbb',        # K% minus BB% — single strongest pitcher metric
    'home_roll_woba',
    'home_roll_hard_pct',   # hard-hit rate allowed (exit velo >95mph)
    'home_prior_xfip',      # prior season xFIP (stable baseline)
    # Away pitcher (faces home batters)
    'away_roll_k_pct',
    'away_roll_bb_pct',
    'away_roll_kbb',
    'away_roll_woba',
    'away_roll_hard_pct',
    'away_prior_xfip',
    # Batting matchup
    'home_leadoff_obp',     # home team leadoff batter trailing OBP
    'away_leadoff_obp',     # away team leadoff batter trailing OBP
    'home_team_k_pct',      # how often home lineup strikes out
    'away_team_k_pct',
    'home_team_woba',
    'away_team_woba',
    # Context
    'park_factor',
    'weather_offense_factor',
]

print(f'Setup complete. Features: {len(FEATURES)}')
print('Seasons:', SEASONS)

## 2 · Load 3-Season Statcast Data

Downloads 2022, 2023, 2024 from Baseball Savant. Each season cached separately.
**First run: ~45-60 min total. Every run after: instant.**

In [ ]:
SEASON_RANGES = {
    2022: [('2022-04-07','2022-04-30'),('2022-05-01','2022-05-31'),
           ('2022-06-01','2022-06-30'),('2022-07-01','2022-07-31'),
           ('2022-08-01','2022-08-31'),('2022-09-01','2022-10-05')],
    2023: [('2023-03-30','2023-04-30'),('2023-05-01','2023-05-31'),
           ('2023-06-01','2023-06-30'),('2023-07-01','2023-07-31'),
           ('2023-08-01','2023-08-31'),('2023-09-01','2023-10-01')],
    2024: [('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
           ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
           ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
}

def load_season(year):
    cache = f'cache/statcast_{year}.csv'
    if os.path.exists(cache):
        print(f'  {year}: loading from cache...')
        df = pd.read_csv(cache, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df
    print(f'  {year}: downloading (~15-20 min)...')
    chunks = []
    for i,(s,e) in enumerate(SEASON_RANGES[year], 1):
        print(f'    [{i}/6] {s} to {e}')
        try: chunks.append(pb.statcast(start_dt=s, end_dt=e)); time.sleep(3)
        except Exception as ex: print(f'    Warning: {ex}')
    df = pd.concat(chunks, ignore_index=True)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df.to_csv(cache, index=False)
    print(f'  {year}: cached ({len(df):,} pitches)')
    return df

print('Loading Statcast data...')
seasons = {yr: load_season(yr) for yr in SEASONS}
sc = pd.concat(seasons.values(), ignore_index=True)
sc['game_date'] = pd.to_datetime(sc['game_date'])
sc = sc.sort_values('game_date').reset_index(drop=True)

print(f'\nCombined: {len(sc):,} pitches | {sc["game_pk"].nunique():,} games')
print(f'Date range: {sc["game_date"].min().date()} to {sc["game_date"].max().date()}')
for yr in SEASONS:
    yr_sc = sc[sc['game_date'].dt.year == yr]
    print(f'  {yr}: {yr_sc["game_pk"].nunique():,} games')

## 3 · FanGraphs Pitcher Quality (xFIP, SIERA, K-BB%)

xFIP and SIERA are **luck-independent ERA estimators** — they strip out HR/FB
rate variance and focus on the things a pitcher actually controls.
We use the **prior season's** values as a stable anchor for each pitcher.
- A pitcher's 2024 xFIP uses their 2023 FanGraphs data (no look-ahead)
- Rookies / new pitchers fall back to league average (4.25)

Merge key: `pybaseball.playerid_reverse_lookup()` maps MLBAM (Statcast) IDs
to FanGraphs `IDfg` — this is the bridge between the two data sources.

In [ ]:
def load_fangraphs_prior_year(seasons):
    """
    For each season in the list, pull the PRIOR year's FanGraphs stats.
    2022 season uses 2021 FanGraphs, 2023 uses 2022, 2024 uses 2023.
    Returns a dict: {season_year: DataFrame with MLBAM pitcher_id + xfip + siera + kbb}
    """
    result = {}
    for yr in seasons:
        prior = yr - 1
        cache = f'cache/fg_pitching_{prior}.csv'
        if os.path.exists(cache):
            df = pd.read_csv(cache)
        else:
            print(f'  Downloading FanGraphs {prior} pitching stats...')
            try:
                df = pb.pitching_stats(prior, prior, qual=30)
                df.to_csv(cache, index=False)
            except Exception as e:
                print(f'  Warning: could not load FG {prior}: {e}')
                result[yr] = pd.DataFrame(columns=['pitcher_mlbam','prior_xfip','prior_siera','prior_kbb'])
                continue

        # Get xFIP, SIERA, K-BB% — handle column name variants
        xfip_col  = next((c for c in df.columns if 'xfip' in c.lower() and '-' not in c.lower()), None)
        siera_col = next((c for c in df.columns if 'siera' in c.lower()), None)
        kbb_col   = next((c for c in df.columns if 'k-bb' in c.lower()), None)
        idfg_col  = next((c for c in df.columns if 'idfg' in c.lower() or c=='IDfg'), None)

        if not idfg_col:
            print(f'  Warning: no IDfg in FG {prior} — skipping')
            result[yr] = pd.DataFrame(columns=['pitcher_mlbam','prior_xfip','prior_siera','prior_kbb'])
            continue

        subset = df[[idfg_col]].copy()
        if xfip_col:  subset['prior_xfip']  = pd.to_numeric(df[xfip_col],  errors='coerce')
        if siera_col: subset['prior_siera'] = pd.to_numeric(df[siera_col], errors='coerce')
        if kbb_col:   subset['prior_kbb']   = pd.to_numeric(df[kbb_col],   errors='coerce')
        subset = subset.rename(columns={idfg_col: 'fg_id'})

        # Map FanGraphs IDfg to MLBAM via reverse lookup
        try:
            fg_ids = subset['fg_id'].dropna().astype(int).tolist()
            lookup = pb.playerid_reverse_lookup(fg_ids, key_type='fangraphs')
            lookup = lookup[['key_fangraphs','key_mlbam']].rename(
                columns={'key_fangraphs':'fg_id','key_mlbam':'pitcher_mlbam'})
            subset = subset.merge(lookup, on='fg_id', how='left')
        except Exception as e:
            print(f'  Warning: playerid_reverse_lookup failed: {e}')
            subset['pitcher_mlbam'] = np.nan

        result[yr] = subset[['pitcher_mlbam','prior_xfip','prior_siera','prior_kbb']].dropna(subset=['pitcher_mlbam'])
        result[yr]['pitcher_mlbam'] = result[yr]['pitcher_mlbam'].astype(int)
        print(f'  FG {prior} (for {yr} season): {len(result[yr])} pitchers with xFIP/SIERA')

    return result

print('Loading FanGraphs pitcher quality metrics...')
fg_by_season = load_fangraphs_prior_year(SEASONS)
print('Done.')

## 4 · Feature Engineering

**Key change from v2:** Rolling stats now use the full game (all innings),
not just inning 1. This gives ~250 PA per 10-start window instead of ~30,
reducing noise by roughly 5x.

In [ ]:
def build_game_log(sc):
    """One row per game: YRFI label, starters, weather, park."""
    inn1 = sc[sc['inning']==1].copy()
    meta = sc.groupby('game_pk').agg(
        game_date=('game_date','first'),
        home_team=('home_team','first'),
        away_team=('away_team','first'),
    ).reset_index()

    def half_runs(g): return (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
    tr = inn1[inn1['inning_topbot']=='Top'].groupby('game_pk').apply(half_runs).reset_index(name='top_runs')
    br = inn1[inn1['inning_topbot']=='Bot'].groupby('game_pk').apply(half_runs).reset_index(name='bot_runs')
    meta = meta.merge(tr,on='game_pk',how='left').merge(br,on='game_pk',how='left')
    meta[['top_runs','bot_runs']] = meta[['top_runs','bot_runs']].fillna(0)
    meta['YRFI'] = ((meta['top_runs']>0)|(meta['bot_runs']>0)).astype(int)

    inn1s = inn1.sort_values(['game_pk','inning_topbot','pitch_number'])
    hp = inn1s[inn1s['inning_topbot']=='Top'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'home_starter_id'})
    ap = inn1s[inn1s['inning_topbot']=='Bot'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'away_starter_id'})
    meta = meta.merge(hp,on='game_pk',how='left').merge(ap,on='game_pk',how='left')

    for col,default in [('wind_speed',5.0),('wind_dir','Calm'),('temperature',72.0)]:
        if col in sc.columns:
            meta = meta.merge(sc.groupby('game_pk')[col].first().reset_index(),on='game_pk',how='left')
        if col not in meta.columns: meta[col]=default
    meta['wind_speed']  = pd.to_numeric(meta['wind_speed'],errors='coerce').fillna(5.0)
    meta['temperature'] = pd.to_numeric(meta['temperature'],errors='coerce').fillna(72.0)
    meta['wind_dir_factor'] = meta['wind_dir'].astype(str).str.strip().map(WIND_DIR_MAP).fillna(0.0)
    meta['weather_offense_factor'] = (meta['wind_dir_factor']*(meta['wind_speed']/10.0)
                                      +(meta['temperature']-72)/30.0)
    meta['park_factor'] = meta['home_team'].map(PARK_FACTORS).fillna(100)/100.0
    meta['game_date'] = pd.to_datetime(meta['game_date'])
    return meta.sort_values('game_date').reset_index(drop=True)


def build_rolling_pitcher_stats(sc, n=ROLLING_N):
    """
    KEY v3 CHANGE: Uses ALL innings (not just inning 1).
    Full-game rolling window gives ~250 PA per window vs ~30 in v2.
    Also adds hard-hit rate (launch_speed >= 95 mph).
    """
    pa = sc[sc['events'].notna()].copy()
    # Hard hit: exit velocity >= 95 mph
    pa['hard_hit'] = (pd.to_numeric(pa.get('launch_speed', np.nan), errors='coerce') >= 95).astype(float)

    gstats = (
        pa.groupby(['pitcher','game_pk','game_date'])
        .agg(
            pa        = ('events','count'),
            k         = ('events', lambda x:(x=='strikeout').sum()),
            bb        = ('events', lambda x:(x=='walk').sum()),
            woba      = ('woba_value','mean'),
            hard_hit  = ('hard_hit','mean'),
        )
        .reset_index()
        .sort_values(['pitcher','game_date'])
    )

    def rs(s): return s.shift(1).rolling(n, min_periods=4).sum()
    def rm(s): return s.shift(1).rolling(n, min_periods=4).mean()

    g = gstats.groupby('pitcher')
    gstats['roll_pa']       = g['pa'].transform(rs)
    gstats['roll_k']        = g['k'].transform(rs)
    gstats['roll_bb']       = g['bb'].transform(rs)
    gstats['roll_woba']     = g['woba'].transform(rm)
    gstats['roll_hard_hit'] = g['hard_hit'].transform(rm)

    gstats['roll_k_pct']   = gstats['roll_k']  / gstats['roll_pa'].replace(0,np.nan)
    gstats['roll_bb_pct']  = gstats['roll_bb'] / gstats['roll_pa'].replace(0,np.nan)
    gstats['roll_kbb']     = gstats['roll_k_pct'] - gstats['roll_bb_pct']
    gstats['roll_hard_pct']= gstats['roll_hard_hit'].fillna(0.35)

    result = gstats[gstats['roll_pa'] >= 20].copy()
    print(f'  Rolling stats (full-game): {len(result):,} pitcher-game rows | '
          f'{result["pitcher"].nunique()} pitchers')
    return result[['pitcher','game_pk','game_date',
                   'roll_k_pct','roll_bb_pct','roll_kbb','roll_woba','roll_hard_pct']]


def build_leadoff_obp(sc, n=ROLLING_BATT):
    """Trailing OBP for leadoff batter in each half-inning (unchanged from v2)."""
    pa = sc[sc['events'].notna()].copy()
    pa['game_date'] = pd.to_datetime(pa['game_date'])
    ON_BASE = {'single','double','triple','home_run','walk','hit_by_pitch'}
    pa['on_base'] = pa['events'].isin(ON_BASE).astype(int)

    inn1_pa = pa[pa['inning']==1].sort_values(['game_pk','inning_topbot','at_bat_number','pitch_number'])
    leadoff = (
        inn1_pa.groupby(['game_pk','inning_topbot'])
        .first().reset_index()
        [['game_pk','inning_topbot','game_date','batter']]
    )
    batter_pa = (
        pa.groupby(['batter','game_pk','game_date'])
        .agg(pa_c=('events','count'), ob_c=('on_base','sum'))
        .reset_index().sort_values(['batter','game_date'])
    )
    batter_pa['trail_pa'] = batter_pa.groupby('batter')['pa_c'].transform(
        lambda x: x.shift(1).rolling(n, min_periods=10).sum())
    batter_pa['trail_ob'] = batter_pa.groupby('batter')['ob_c'].transform(
        lambda x: x.shift(1).rolling(n, min_periods=10).sum())
    batter_pa['trail_obp'] = batter_pa['trail_ob'] / batter_pa['trail_pa'].replace(0,np.nan)

    lo = leadoff.merge(batter_pa[['batter','game_pk','trail_obp']], on=['batter','game_pk'], how='left')
    lo['trail_obp'] = lo['trail_obp'].fillna(0.320)

    top = lo[lo['inning_topbot']=='Top'][['game_pk','trail_obp']].rename(columns={'trail_obp':'away_leadoff_obp'})
    bot = lo[lo['inning_topbot']=='Bot'][['game_pk','trail_obp']].rename(columns={'trail_obp':'home_leadoff_obp'})
    result = top.merge(bot, on='game_pk', how='outer').fillna(0.320)
    print(f'  Leadoff OBP: {len(result):,} games')
    return result


def build_team_batting(sc):
    """Season-level team batting stats (stable context features)."""
    pa = sc[sc['events'].notna()].copy()
    pa['batting_team'] = np.where(pa['inning_topbot']=='Top', pa['away_team'], pa['home_team'])
    pa['season'] = pd.to_datetime(pa['game_date']).dt.year
    return pa.groupby(['batting_team','season']).agg(
        team_woba  =('woba_value','mean'),
        team_k_pct =('events',lambda x:(x=='strikeout').sum()/len(x)),
    ).reset_index()


print('Building features...')
game_log    = build_game_log(sc)
rolling     = build_rolling_pitcher_stats(sc)
leadoff     = build_leadoff_obp(sc)
team_stats  = build_team_batting(sc)
print(f'  Game log: {len(game_log):,} games | YRFI rate: {game_log["YRFI"].mean():.1%}')

## 5 · Assemble Modeling Dataset

In [ ]:
def assemble(game_log, rolling, leadoff, team_stats, fg_by_season):
    df = game_log.copy()
    df['season'] = df['game_date'].dt.year

    # Home pitcher rolling stats
    hr = rolling.rename(columns={
        'roll_k_pct':'home_roll_k_pct','roll_bb_pct':'home_roll_bb_pct',
        'roll_kbb':'home_roll_kbb','roll_woba':'home_roll_woba',
        'roll_hard_pct':'home_roll_hard_pct'
    })
    df = df.merge(
        hr[['pitcher','game_pk','home_roll_k_pct','home_roll_bb_pct',
            'home_roll_kbb','home_roll_woba','home_roll_hard_pct']],
        left_on=['home_starter_id','game_pk'], right_on=['pitcher','game_pk'],
        how='inner'
    ).drop(columns=['pitcher'])

    # Away pitcher rolling stats
    ar = rolling.rename(columns={
        'roll_k_pct':'away_roll_k_pct','roll_bb_pct':'away_roll_bb_pct',
        'roll_kbb':'away_roll_kbb','roll_woba':'away_roll_woba',
        'roll_hard_pct':'away_roll_hard_pct'
    })
    df = df.merge(
        ar[['pitcher','game_pk','away_roll_k_pct','away_roll_bb_pct',
            'away_roll_kbb','away_roll_woba','away_roll_hard_pct']],
        left_on=['away_starter_id','game_pk'], right_on=['pitcher','game_pk'],
        how='inner'
    ).drop(columns=['pitcher'])

    # FanGraphs prior-year xFIP — per season
    fg_all = []
    for yr, fg in fg_by_season.items():
        if len(fg) == 0: continue
        fg = fg.copy(); fg['season'] = yr
        fg_all.append(fg)

    if fg_all:
        fg_df = pd.concat(fg_all, ignore_index=True)
        # Home pitcher xFIP
        fg_h = fg_df.rename(columns={'pitcher_mlbam':'pid','prior_xfip':'home_prior_xfip','prior_siera':'home_prior_siera','prior_kbb':'home_prior_kbb_fg'})
        df = df.merge(fg_h[['pid','season','home_prior_xfip']],
                      left_on=['home_starter_id','season'], right_on=['pid','season'], how='left').drop(columns=['pid'])
        # Away pitcher xFIP
        fg_a = fg_df.rename(columns={'pitcher_mlbam':'pid','prior_xfip':'away_prior_xfip','prior_siera':'away_prior_siera','prior_kbb':'away_prior_kbb_fg'})
        df = df.merge(fg_a[['pid','season','away_prior_xfip']],
                      left_on=['away_starter_id','season'], right_on=['pid','season'], how='left').drop(columns=['pid'])
    else:
        df['home_prior_xfip'] = 4.25  # league average fallback
        df['away_prior_xfip'] = 4.25

    # Fill missing xFIP with league average
    df['home_prior_xfip'] = df['home_prior_xfip'].fillna(4.25)
    df['away_prior_xfip'] = df['away_prior_xfip'].fillna(4.25)

    # Leadoff OBP
    df = df.merge(leadoff, on='game_pk', how='left')
    df['away_leadoff_obp'] = df.get('away_leadoff_obp', 0.320).fillna(0.320)
    df['home_leadoff_obp'] = df.get('home_leadoff_obp', 0.320).fillna(0.320)

    # Team batting stats (season-specific)
    ts = team_stats.rename(columns={'batting_team':'t','team_woba':'away_team_woba','team_k_pct':'away_team_k_pct'})
    ts2= team_stats.rename(columns={'batting_team':'t','team_woba':'home_team_woba','team_k_pct':'home_team_k_pct'})
    df = df.merge(ts, left_on=['away_team','season'], right_on=['t','season'], how='left').drop(columns=['t'])
    df = df.merge(ts2,left_on=['home_team','season'], right_on=['t','season'], how='left').drop(columns=['t'])

    return df.sort_values('game_date').reset_index(drop=True)


print('Assembling dataset...')
df = assemble(game_log, rolling, leadoff, team_stats, fg_by_season)
df_clean = df.dropna(subset=FEATURES+['YRFI']).copy()

print(f'\nFull dataset  : {len(df):,} games')
print(f'After dropna  : {len(df_clean):,} games')
print(f'YRFI rate     : {df_clean["YRFI"].mean():.1%}')
print(f'Date range    : {df_clean["game_date"].min().date()} to {df_clean["game_date"].max().date()}')
print(f'\nPer season:')
for yr in SEASONS:
    yr_df = df_clean[df_clean['game_date'].dt.year==yr]
    print(f'  {yr}: {len(yr_df):,} games | YRFI: {yr_df["YRFI"].mean():.1%}')

## 6 · Backtest — 60/40 Chronological Split

**v3 model:** Single unified logistic regression with calibrated probabilities.
CalibratedClassifierCV (isotonic regression) transforms raw logit scores
into proper probabilities — so a 65% prediction actually means ~65% of
those games were YRFI in the training set.

Threshold tuned on training set.

In [ ]:
df_s = df_clean.sort_values('game_date').reset_index(drop=True)
split = int(len(df_s)*0.60)
train, test = df_s.iloc[:split].copy(), df_s.iloc[split:].copy()

print(f'Train: {len(train):,} games  ({train["game_date"].min().date()} to {train["game_date"].max().date()})')
print(f'Test:  {len(test):,} games   ({test["game_date"].min().date()} to {test["game_date"].max().date()})')
print(f'Train YRFI: {train["YRFI"].mean():.1%}  |  Test YRFI: {test["YRFI"].mean():.1%}')

# Scale
scaler = StandardScaler()
X_tr = scaler.fit_transform(train[FEATURES])
X_te = scaler.transform(test[FEATURES])

# Base logistic regression
base = LogisticRegression(C=0.5, max_iter=2000, class_weight='balanced', random_state=42)

# Calibrated wrapper — isotonic calibration on a held-out fold of training data
model = CalibratedClassifierCV(base, method='isotonic', cv=5)
model.fit(X_tr, train['YRFI'])

proba  = model.predict_proba(X_te)[:,1]
proba_tr = model.predict_proba(X_tr)[:,1]

# Tune threshold on training set
best_t, best_acc = 0.5, 0.0
for t in np.arange(0.35, 0.75, 0.01):
    a = accuracy_score(train['YRFI'], (proba_tr>=t).astype(int))
    if a > best_acc: best_acc, best_t = a, t

preds  = (proba >= best_t).astype(int)
acc    = accuracy_score(test['YRFI'], preds)
auc    = roc_auc_score(test['YRFI'], proba)
brier  = brier_score_loss(test['YRFI'], proba)

results = test[['game_date','home_team','away_team','YRFI']].copy()
results['yrfi_probability'] = proba
results['predicted_YRFI']   = preds
results['correct']          = (preds == test['YRFI'].values).astype(int)

# Feature importance from underlying base estimator
try:
    coef = model.calibrated_classifiers_[0].estimator.coef_[0]
except:
    coef = np.zeros(len(FEATURES))
fi = pd.DataFrame({'feature':FEATURES,'coef':coef}).sort_values('coef',key=abs,ascending=False)

print(f'\nThreshold (tuned): {best_t:.2f}')
print(f'Prob spread  — mean: {proba.mean():.3f}  std: {proba.std():.3f}  '
      f'min: {proba.min():.3f}  max: {proba.max():.3f}')
print(f'\n{"="*50}')
print(f'  v1 accuracy    : 54.8%')
print(f'  v2 accuracy    : 51.3%')
print(f'  v3 accuracy    : {acc:.1%}')
print(f'  ROC-AUC        : {auc:.3f}  (0.5=random, 1.0=perfect)')
print(f'  Brier score    : {brier:.4f}  (lower=better, 0.25=random)')
print(f'{"="*50}')
print(f'\n{classification_report(test["YRFI"], preds, target_names=["NRFI","YRFI"], digits=3)}')

## 7 · Results Dashboard

**Read Panel 8 first** (bottom-right — confidence tier accuracy).
If strong conviction tiers hit 60%+, the model is filtering signal from noise.

In [ ]:
sns.set_theme(style='darkgrid', palette='muted', font_scale=0.95)
fig = plt.figure(figsize=(22,16))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(
    f'NRFI/YRFI Model v3 — 3-Year Dataset, Full-Game Rolling, xFIP, Calibrated\n'
    f'v3: {acc:.1%}  |  v2: 51.3%  |  v1: 54.8%  |  ROC-AUC: {auc:.3f}  |  Brier: {brier:.4f}',
    fontsize=14, fontweight='bold', color='white', y=0.98
)
gs = gridspec.GridSpec(3,3,figure=fig,hspace=0.50,wspace=0.38)
B,P,G,R,O,BG,T='#2196F3','#9C27B0','#4CAF50','#F44336','#FF9800','#1a1d27','#e0e0e0'

def sax(ax,title):
    ax.set_facecolor(BG);ax.set_title(title,fontsize=11,fontweight='bold',color=T,pad=8)
    ax.tick_params(colors=T);ax.xaxis.label.set_color(T);ax.yaxis.label.set_color(T)
    [s.set_edgecolor('#444') for s in ax.spines.values()]

# 1. Rolling accuracy vs v1 and v2
ax1=fig.add_subplot(gs[0,:2])
r=results.sort_values('game_date').copy();r['ra']=r['correct'].rolling(30,min_periods=10).mean()
ax1.fill_between(r['game_date'],r['ra'],alpha=0.2,color=B)
ax1.plot(r['game_date'],r['ra'],color=B,lw=2,label='30-game rolling')
ax1.axhline(acc,  color=R,    ls='--',lw=1.8,label=f'v3 Overall: {acc:.1%}')
ax1.axhline(0.548,color=O,    ls='--',lw=1.2,alpha=0.8,label='v1: 54.8%')
ax1.axhline(0.513,color=P,    ls='--',lw=1.2,alpha=0.8,label='v2: 51.3%')
ax1.axhline(0.50, color='gray',ls=':',alpha=0.6,label='50% baseline')
ax1.set_ylim(0.30,0.85);ax1.set_ylabel('Accuracy',color=T)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax1,'NRFI/YRFI v3 — 30-Game Rolling Accuracy')

# 2. Feature importance
ax2=fig.add_subplot(gs[0,2])
ft=fi.head(10).iloc[::-1]
ax2.barh(ft['feature'],ft['coef'],color=[G if c>0 else R for c in ft['coef']],height=0.6)
ax2.axvline(0,color='white',lw=0.8);ax2.set_xlabel('Coefficient',color=T)
sax(ax2,'v3 Feature Importance\n(+YRFI  -NRFI)')

# 3. Confusion matrix
ax3=fig.add_subplot(gs[1,0])
cm_=confusion_matrix(results['YRFI'],results['predicted_YRFI'])
sns.heatmap(cm_,annot=True,fmt='d',cmap='Blues',ax=ax3,
            xticklabels=['Pred NRFI','Pred YRFI'],yticklabels=['Act NRFI','Act YRFI'],
            linewidths=0.5,linecolor='#333',annot_kws={'size':13,'weight':'bold'})
ax3.set_facecolor(BG);ax3.tick_params(colors=T)
ax3.set_title('Confusion Matrix v3',fontsize=11,fontweight='bold',color=T,pad=8)

# 4. Probability separation — key diagnostic
ax4=fig.add_subplot(gs[1,1])
results[results['YRFI']==0]['yrfi_probability'].hist(ax=ax4,bins=30,alpha=0.65,color=B,label='Actual NRFI',density=True)
results[results['YRFI']==1]['yrfi_probability'].hist(ax=ax4,bins=30,alpha=0.65,color=O,label='Actual YRFI',density=True)
ax4.axvline(best_t,color='white',ls='--',lw=1.5,label=f'Threshold {best_t:.2f}')
ax4.set_xlabel('Predicted YRFI Probability',color=T)
ax4.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax4,'Probability Separation\n(v1 had both distributions on top of each other)')

# 5. Calibration curve
ax5=fig.add_subplot(gs[1,2])
bins_cal=np.linspace(0,1,11)
res2=results.copy()
res2['prob_bin']=pd.cut(res2['yrfi_probability'],bins=bins_cal)
cal=res2.groupby('prob_bin',observed=True)['YRFI'].agg(actual_rate='mean',n='count').reset_index()
cal['bin_mid']=[i.mid for i in cal['prob_bin']]
ax5.plot([0,1],[0,1],color='gray',ls='--',alpha=0.6,label='Perfect calibration')
ax5.scatter(cal['bin_mid'],cal['actual_rate'],s=cal['n']/5,color=B,alpha=0.8,label='v3 calibration')
ax5.plot(cal['bin_mid'],cal['actual_rate'],color=B,lw=1.5,alpha=0.7)
ax5.set_xlabel('Predicted probability',color=T);ax5.set_ylabel('Actual YRFI rate',color=T)
ax5.set_xlim(0,1);ax5.set_ylim(0,1)
ax5.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax5,'Calibration Curve\n(dots on diagonal = well-calibrated probabilities)')

# 6. Monthly accuracy
ax6=fig.add_subplot(gs[2,0])
ORDER=['Mar','Apr','May','Jun','Jul','Aug','Sep','Oct']
r2=results.copy();r2['month']=pd.to_datetime(r2['game_date']).dt.strftime('%b')
mo=r2.groupby('month')['correct'].agg(acc_m='mean',n='count').reset_index()
mo['month']=pd.Categorical(mo['month'],categories=ORDER,ordered=True)
mo=mo.sort_values('month').dropna(subset=['month'])
bars=ax6.bar(mo['month'],mo['acc_m']*100,
             color=[G if v>0.55 else O if v>0.50 else R for v in mo['acc_m']],edgecolor='none')
ax6.axhline(50,color='gray',ls=':',alpha=0.6)
ax6.axhline(54.8,color=O,ls='--',lw=1.2,alpha=0.8,label='v1: 54.8%')
ax6.set_ylim(35,80);ax6.set_ylabel('Accuracy %',color=T)
for bar,row in zip(bars,mo.itertuples()):
    ax6.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,
             f'{row.acc_m*100:.0f}%\nn={row.n}',ha='center',va='bottom',color=T,fontsize=8)
ax6.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax6,'v3 Accuracy by Month\n(Green>55  Orange>50  Red<50)')

# 7. Year-over-year accuracy
ax7=fig.add_subplot(gs[2,1])
r3=results.copy();r3['year']=pd.to_datetime(r3['game_date']).dt.year.astype(str)
yr_acc=r3.groupby('year')['correct'].agg(a='mean',n='count').reset_index()
bars2=ax7.bar(yr_acc['year'],yr_acc['a']*100,
              color=[G if v>0.55 else O if v>0.50 else R for v in yr_acc['a']],edgecolor='none',width=0.5)
ax7.axhline(50,color='gray',ls=':',alpha=0.6)
ax7.axhline(54.8,color=O,ls='--',lw=1.2,alpha=0.8,label='v1 overall')
ax7.set_ylim(35,80);ax7.set_ylabel('Accuracy %',color=T)
for bar,row in zip(bars2,yr_acc.itertuples()):
    ax7.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,
             f'{row.a*100:.0f}%\nn={row.n}',ha='center',va='bottom',color=T,fontsize=9)
ax7.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax7,'v3 Accuracy by Year\n(consistency across seasons = robust signal)')

# 8. Confidence tier — THE KEY DIAGNOSTIC
ax8=fig.add_subplot(gs[2,2])
bins_=[0,0.35,0.45,0.55,0.65,1.01]
lbls=['<35\nStr NRFI','35-45\nLn NRFI','45-55\nToss-up','55-65\nLn YRFI','>65\nStr YRFI']
results['tier']=pd.cut(results['yrfi_probability'],bins=bins_,labels=lbls)
ta=results.groupby('tier',observed=True)['correct'].agg(a='mean',n='count').reset_index()
bars3=ax8.bar(ta['tier'],ta['a']*100,
              color=[G if v>0.55 else O if v>0.50 else R for v in ta['a']],edgecolor='none')
for bar,row in zip(bars3,ta.itertuples()):
    ax8.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,
             f'{row.a*100:.0f}%\nn={row.n}',ha='center',va='bottom',color=T,fontsize=8)
ax8.axhline(50,color='gray',ls=':',alpha=0.6)
ax8.set_ylim(35,80);ax8.set_ylabel('Accuracy %',color=T)
sax(ax8,'Accuracy by Confidence Tier\n(outer bars 60%+ = model knows when it is right)')

plt.savefig('nrfi_model_v3_results.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()
print('Chart saved.')

## 8 · Save Results

In [ ]:
results.to_csv('nrfi_v3_backtest.csv', index=False)
print(f'Saved: nrfi_v3_backtest.csv  ({len(results):,} rows)')
print(f'Prob spread  : {results["yrfi_probability"].std():.4f}  (v2 was 0.042, want >0.10)')
print(f'Overall acc  : {acc:.1%}')
print(f'ROC-AUC      : {auc:.3f}')
print(f'Brier score  : {brier:.4f}  (0.25=random, lower=better)')
print()
print('Monthly breakdown:')
r2=results.copy();r2['month']=pd.to_datetime(r2['game_date']).dt.strftime('%b')
ORDER=['Mar','Apr','May','Jun','Jul','Aug','Sep','Oct']
for m,row in r2.groupby('month')['correct'].agg(a='mean',n='count').iterrows():
    print(f'  {m:<5} {row["a"]*100:>5.1f}%  n={int(row["n"])}')
print()
print('Confidence tier breakdown:')
for row in results.groupby('tier',observed=True)['correct'].agg(a='mean',n='count').itertuples():
    flag = 'GOOD' if row.a>0.58 else 'OK' if row.a>0.52 else 'WEAK'
    print(f'  {str(row.Index):<20} {row.a*100:>5.1f}%  n={row.n}  {flag}')
try:
    from google.colab import files
    files.download('nrfi_v3_backtest.csv')
    files.download('nrfi_model_v3_results.png')
    print('Downloads triggered.')
except ImportError:
    print('(Not in Colab - files saved locally)')